# GeoSPSR Comparison

## Imports

In [1]:
from functools import partial
import numpy as np
import jax.numpy as jnp
from jax import vmap, random

from geospsr.kernels import Matern32Kernel, ProductKernel
from geospsr.utils import trunc_Zd, periodic_stationary_interpolator
from geospsr.gp import (poisson_cross_covariances, compute_mean,
                        f_eigenvalues_from_v_eigenvalues)

from skimage.measure import marching_cubes
import open3d as o3d
from pyprojroot import here
from huggingface_hub import snapshot_download

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Download point cloud data

In [2]:
DATA_DIR = here("data/ppsurf")

# https://huggingface.co/datasets/perler/ppsurf
if not (DATA_DIR / "famous_noisefree").exists():
    snapshot_download(
        "perler/ppsurf",
        repo_type="dataset",
        allow_patterns=["famous_*/*"],
        local_dir=DATA_DIR
    )

## Functions

In [3]:
VARIANTS = ["famous_noisefree", "famous_original", "famous_sparse",
            "famous_dense", "famous_extra_noisy"]
SHAPES = ["xyzrgb_statuette"]

def load_points(variant: str, shape: str) -> np.ndarray:
    path = DATA_DIR / variant / "04_pts_vis" / f"{shape}.xyz.ply"
    return np.asarray(o3d.io.read_point_cloud(str(path)).points)

def estimate_normals(points: np.ndarray, knn_k=30, orient_k=20) -> np.ndarray:
    pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(points))
    pcd.estimate_normals(o3d.geometry.KDTreeSearchParamKNN(knn_k))
    pcd.orient_normals_consistent_tangent_plane(orient_k)
    center = np.mean(points, axis=0)
    normals = np.asarray(pcd.normals)
    
    dots = np.sum((points - center) * normals, axis=1)
    if np.mean(dots > 0) < 0.5:
        normals = -normals
    return normals

def normalize_points(v):
    lo, hi = v.min(0), v.max(0)
    center = (lo + hi) / 2
    scale = (hi - lo).max()
    return (v - center) / scale, center, scale

def denormalize_points(v, center, scale):
    return v * scale + center

def get_gt_pcd(variant: str, shape: str, N: int) -> o3d.geometry.PointCloud:
    path = DATA_DIR / variant / "03_meshes" / f"{shape}.ply"
    gt_mesh = o3d.io.read_triangle_mesh(str(path))
    gt_mesh.compute_triangle_normals()
    gt_mesh.compute_vertex_normals()
    gt_pcd = gt_mesh.sample_points_uniformly(N, True)
    return gt_pcd


## Kernel boilerplate

In [4]:
sigma = 0.02
lengthscale = 1e-2
variance = 0.1
truncation_n = 50
amortization_density = 50

k = Matern32Kernel(lengthscale, variance)
k_v = ProductKernel(k, k, k)

k_v_eigenvectors = trunc_Zd(truncation_n)
k_v_eigenvalues = vmap(k_v.spectral_density)(k_v_eigenvectors) ** 0.5

k_fv_expensive = partial(
    poisson_cross_covariances,
    eigenvectors=k_v_eigenvectors,
    eigenvalues=k_v_eigenvalues,
)
k_fv = periodic_stationary_interpolator(
    k_fv_expensive, 3, amortization_density, exponent=5, verbose=True
)

k_f_eigenvalues = f_eigenvalues_from_v_eigenvalues(
    k_v_eigenvectors, k_v_eigenvalues)
k_f_variance = (k_f_eigenvalues**2).sum()



Creating interpolator:   0%|          | 0/977 [00:00<?, ?it/s]

## Reconstruction

In [ ]:
OUT_DIR = here("data/results/comparison")
OUT_DIR.mkdir(parents=True, exist_ok=True)


def sdd_params_for(n_points, bs=128):
    """
    SDD settings that converge for this kernel (variance 0.1 per axis, ls 0.01).
    The stochastic gradient is scaled by N/bs, so the stable lr shrinks ~1/N;
    40 epochs gave residual < 1e-3 and alpha within 0.2% of Cholesky on all 5 variants.
    """
    return {"key": random.key(0), "bs": bs, "lr": 6000 / n_points,
            "iterations": int(np.ceil(40 * n_points / bs))}


def geospsr_mesh(x_data, v_data, lin, center, scale, sdd_params=None):
    """GeoSPSR mean surface -> mesh in dataset coords; sdd_params=None -> Cholesky"""
    res = len(lin)
    X, Y, Z = np.meshgrid(lin, lin, lin, indexing="ij")
    x_grid = np.stack([X, Y, Z], -1).reshape(-1, 3)

    # Concatenated grid points and data points
    # to avoid calling compute_mean function twice.
    query = jnp.concatenate([x_data, x_grid])
    mean = compute_mean(
        query, x_data, v_data, k_v, k_fv, sigma,
        bs=int(np.clip(2**30 // (len(x_data) * 120), 32, 4096)),
        sdd_params=sdd_params, verbose=True
    )

    iso = mean[:len(x_data)].mean()
    vol = np.array(mean[len(x_data):]).reshape(res, res, res)
    step = lin[1] - lin[0]
    verts, faces, _, _ = marching_cubes(vol, iso, spacing=(step, step, step))
    verts = verts + lin[0]
    verts = denormalize_points(verts / np.pi, center, scale)

    return o3d.geometry.TriangleMesh(
        o3d.utility.Vector3dVector(verts),
        o3d.utility.Vector3iVector(faces)
    )


for variant in VARIANTS:
    for shape in SHAPES:

        res = 64
        half = np.pi / 2 * 1.1
        lin = np.linspace(-half, half, res)

        p = load_points(variant, shape)
        p, center, scale = normalize_points(p)
        p = p * np.pi
        n = estimate_normals(p)

        x_data = jnp.array(p)
        v_data = jnp.array(n)

        for solver, sdd_params in [("cholesky", None),
                                   ("sdd", sdd_params_for(len(x_data)))]:
            path = OUT_DIR / f"{shape}_{variant}_geospsr_{solver}.ply"
            if not path.exists():
                mesh = geospsr_mesh(x_data, v_data, lin, center, scale, sdd_params)
                ok = o3d.io.write_triangle_mesh(str(path), mesh)
                assert ok, f"failed to write {path}"

        path = OUT_DIR / f"{shape}_{variant}_screened_psr.ply"    
        if not path.exists():
            p = load_points(variant, shape)
            pcd = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(p))
            pcd.normals = o3d.utility.Vector3dVector(estimate_normals(p))
            mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
                pcd, depth=8)
            ok = o3d.io.write_triangle_mesh(str(path), mesh)
            assert ok, f"failed to write {path}"

## Visualization

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def mesh_trace(mesh, gt_scene=None, max_faces=50_000):
    """
    o3d mesh -> plotly Mesh3d (decimated so the browser stays responsive)
    gt_scene: RaycastingScene of the GT mesh -> vertices colored by distance to GT;
              None -> plain gray (used for the GT itself)
    """
    if len(mesh.triangles) > max_faces:
        mesh = mesh.simplify_quadric_decimation(max_faces)
    v, f = np.asarray(mesh.vertices), np.asarray(mesh.triangles)
    if gt_scene is None:
        kw = dict(color="#b4b4b4")   # mid gray: lightgray saturates to white under lighting
    else:
        # distance after decimation, so there is one value per plotted vertex
        dist = gt_scene.compute_distance(o3d.core.Tensor(v.astype(np.float32))).numpy()
        kw = dict(intensity=dist, intensitymode="vertex", coloraxis="coloraxis")
    return go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2],
                     i=f[:, 0], j=f[:, 1], k=f[:, 2],
                     flatshading=True, hoverinfo="skip",
                     # plotly's default light sits at +y, i.e. behind the mesh for our
                     # camera (-y): visible faces got ambient only -> no shading.
                     # Put the light on the camera side, above, and lower the ambient.
                     lightposition=dict(x=5e4, y=-1e5, z=1e5),
                     lighting=dict(ambient=0.3, diffuse=0.9, specular=0.15,
                                   roughness=0.5, fresnel=0.1), **kw)


def show_comparison(variant, shape, mesh_paths, cmax_ratio=0.01, height=450):
    """
    one row: GT (gray) | each method colored by distance to GT
    cmax_ratio: color scale upper bound as a fraction of the GT bbox diagonal
    """
    gt = o3d.io.read_triangle_mesh(
        str(DATA_DIR / variant / "03_meshes" / f"{shape}.ply"))
    meshes = {"GT": gt} | {name: o3d.io.read_triangle_mesh(
        str(p)) for name, p in mesh_paths.items()}

    gt_scene = o3d.t.geometry.RaycastingScene()
    gt_scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(gt))

    fig = make_subplots(rows=1, cols=len(meshes), subplot_titles=list(meshes),
                        specs=[[{"type": "scene"}] * len(meshes)],
                        horizontal_spacing=0.01)
    for col, (name, mesh) in enumerate(meshes.items(), start=1):
        fig.add_trace(mesh_trace(mesh, None if name == "GT" else gt_scene),
                      row=1, col=col)

    # same viewpoint for every panel
    camera = dict(eye=dict(x=0, y=-1.8, z=0.3))
    fig.update_scenes(aspectmode="data", camera=camera,
                      xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)
    # shared color scale, fixed so that methods are comparable
    diag = np.linalg.norm(np.ptp(np.asarray(gt.vertices), 0))
    fig.update_layout(coloraxis=dict(colorscale="Viridis", cmin=0, cmax=cmax_ratio * diag,
                                     colorbar=dict(title="dist to GT")))
    fig.update_layout(title=f"{shape} / {variant}", height=height, showlegend=False,
                      margin=dict(l=0, r=0, t=60, b=0))
    return fig


for variant in VARIANTS:
    for shape in SHAPES:
        show_comparison(variant, shape, {
            "GeoSPSR (Cholesky)": OUT_DIR / f"{shape}_{variant}_geospsr_cholesky.ply",
            "GeoSPSR (SDD)": OUT_DIR / f"{shape}_{variant}_geospsr_sdd.ply",
            "Screened PSR": OUT_DIR / f"{shape}_{variant}_screened_psr.ply",
        }).show()

## Conclusion

In this setup, GeoSPSR's mean surfaces look noticeably smoother than the GT and fine details of the statuette are lost.

GeoSPSR's main strengths also lie outside what this comparison measures: a posterior over surfaces (variance, samples) and evaluation at arbitrary query points without a global grid.
